# ⚙️ Scikit-Learn Production Pipeline Mastery (Google Colab Notebook)
> **বাংলা টিউটোরিয়াল:** Scikit-Learn-এর পাইপলাইন, কলাম ট্রান্সফরমার ও হাইপারপ্যারামিটার টিউনিং হ্যান্ডস-অন প্র্যাকটিস।

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("Scikit-Learn loaded successfully!")

## ১. ডেটাসেট তৈরি ও ট্রেন-টেস্ট স্প্লিট

In [ ]:
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age': np.random.choice([22, 28, 35, 42, 58, np.nan], size=n),
    'balance': np.random.uniform(500, 50000, size=n),
    'education': np.random.choice(['Secondary', 'Tertiary', 'Primary', np.nan], size=n),
    'loan': np.random.choice(['yes', 'no'], size=n),
    'subscribed': np.random.choice([0, 1], size=n, p=[0.8, 0.2])
})

X = df.drop('subscribed', axis=1)
y = df['subscribed']

# সর্বদা প্রিপ্রসেসিংয়ের পূর্বে স্প্লিট করতে হবে (ডাটা লিকেজ রোধে)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

## ২. কলাম ট্রান্সফরমার ও প্রিপ্রসেসিং পাইপলাইন

In [ ]:
num_cols = ['age', 'balance']
cat_cols = ['education', 'loan']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

pipeline

## ৩. ট্রেনিং, ক্রস-ভ্যালিডেশন ও হাইপারপ্যারামিটার টিউনিং

In [ ]:
param_grid = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0],
    'classifier__solver': ['lbfgs', 'liblinear']
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print(f"Best Cross-Validation ROC-AUC: {grid.best_score_:.4f}")

# টেস্ট সেটে ফাইনাল প্রেডিকশন
y_pred = grid.predict(X_test)
print("\nClassification Report on Test Data:\n")
print(classification_report(y_test, y_pred))